In [1]:
import os
import pandas as pd
import numpy as np
import scanpy as sc
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import radius_neighbors_graph

import seaborn as sns

from matplotlib.pyplot import rc_context
import matplotlib.colors as mcolors
from matplotlib.colors import Normalize
from matplotlib.colorbar import ColorbarBase

import pySTIM as pst

import warnings
warnings.filterwarnings('ignore')
cmap = mcolors.LinearSegmentedColormap.from_list('WhRd',['#ffffff', "#fffacd", "red", "darkred"], N=256)  

In [6]:
adata=CRC_subcluster_annot.copy()

In [22]:
adata.obs["x"] = adata.obsm["spatial"][:, 0]
adata.obs["y"] = adata.obsm["spatial"][:, 1]

In [ ]:
adata

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.neighbors import radius_neighbors_graph
import seaborn as sns
import matplotlib.pyplot as plt

radius = 55/2                   
n_neighborhoods = 8           
random_state = 0
batch_size = 10_000            
pseudocount = 1e-6            

batches = (
    adata.obs['batch'].astype(str).astype('category').cat.categories.tolist()
    if pd.api.types.is_categorical_dtype(adata.obs['batch'])
    else sorted(adata.obs['batch'].astype(str).unique().tolist())
)

neighbors_dict = {}

for b in batches:
    print(f"[batch {b}] subsetting & building radius graph...")
    tmp = adata[adata.obs['batch'].astype(str) == b].copy()

    xy = tmp.obs[['x', 'y']].to_numpy()

    ct = tmp.obs['cell_type_revised'].astype('category')
    one_hot = pd.get_dummies(ct)                    

    G = radius_neighbors_graph(
        xy, radius=radius, mode='connectivity', include_self=False
    )                                               

    neigh_counts = G.dot(one_hot.values)           

    df = pd.DataFrame(neigh_counts, index=tmp.obs_names, columns=one_hot.columns)
    row_sums = df.sum(axis=1)
    df = df.div(row_sums.replace(0, np.nan), axis=0).fillna(0.0)

    df['cell_type_revised'] = ct.to_numpy()
    df['batch'] = b
    df['x'] = tmp.obs['x'].to_numpy()
    df['y'] = tmp.obs['y'].to_numpy()

    neighbors_dict[b] = df

all_neighbors_df = pd.concat(neighbors_dict.values(), axis=0)

cell_type_cols = [c for c in all_neighbors_df.columns
                  if c not in ['cell_type_revised', 'batch', 'x', 'y']]

values = all_neighbors_df[cell_type_cols].to_numpy()

print("Clustering neighborhoods (MiniBatchKMeans)...")
kmeans = MiniBatchKMeans(
    n_clusters=n_neighborhoods, random_state=random_state, batch_size=batch_size
)
labels = kmeans.fit_predict(values)
all_neighbors_df['niche'] = labels

niche_profile = (
    all_neighbors_df.groupby('niche')[cell_type_cols].mean().sort_index()
)  

tissue_avgs = all_neighbors_df[cell_type_cols].mean(axis=0)

fc = np.log2((niche_profile + pseudocount).div(tissue_avgs + pseudocount, axis=1))

fc.index = [f"CN_{i}" for i in fc.index]

sns.set(font_scale=1.2)
g = sns.clustermap(
    fc, cmap="RdYlBu_r", center=0, row_cluster=False, col_cluster=True,
    figsize=(10, 8), cbar_kws={"shrink": 0.8, "label": "Enrichment scores"}
)
g.ax_heatmap.set_xlabel("")
g.ax_heatmap.set_ylabel("")
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90)
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)
plt.show()

topN = 5
top_enriched = (
    fc.apply(lambda s: s.sort_values(ascending=False).head(topN).index.tolist(), axis=1)
)
